(parameters)=

# 参数类的使用

`Gal3d` 使用 {py:class}`Parameters <gal3d.optimization.parameter.Parameters>` 类来管理和记录参数。  

## 基本使用

我们可以如下定义一组参数：

In [1]:
from gal3d.optimization.parameter import Parameters

param = Parameters(a=3,b=2,c=1)

每个参数的值都是一个 {py:class}`Parameter <gal3d.optimization.parameter.Parameter>` 对象。  
参数的下界 (`lb`) 和上界 (`ub`) 可分别指定，若未指定则默认为无穷小和无穷大。

In [2]:
print(param['a'].__class__.__name__, 
      param['a'], param['a'].lb,param['a'].ub)

Parameter 3.0 -inf inf


可以直接修改参数的 `lb` 和 `ub`：

In [ ]:
param['a'].lb = 0

也可以通过 `set_ub` 和 `set_lb` 方法批量设置：

In [4]:
param.set_ub(b=0,c=0)
print(param['b'].ub,param['c'].ub)

0.0 0.0


## 添加额外信息与衍生参数

通过 `add_derived` 方法可以添加衍生参数，所有衍生参数的名称可通过 `derived_keys` 获取：

In [5]:
param.add_derived("eps_ab",lambda p: 1 - p['b']/p['a'])
print(param.derived_keys(),"eps_ab: ", param['eps_ab'])

dict_keys(['eps_ab']) eps_ab:  0.33333333333333337


通过 `add_info` 方法可添加其他信息，所有信息名称可通过 `info_keys` 获取：

In [6]:
param.add_info(mass = 10)
print(param.info_keys(),"mass: ", param['mass'])

dict_keys(['mass']) mass:  10


基本参数名称列表可通过 `parameter_keys` 获取，这些参数均为 {py:class}`Parameter <gal3d.optimization.parameter.Parameter>` 对象：

In [7]:
param.parameter_keys()

dict_keys(['a', 'b', 'c'])

## 添加参数约束

可以通过 `add_equal_constraints` 方法为基本参数添加等式约束：

In [8]:
param = Parameters(a=3,b=2,c=1)
param.set_lb(a=0.3,b=0.2,c=0.1)

param.add_equal_constraints(c = lambda p: p['b'])
param['c']

2.0

被添加等式约束的参数会从 `parameter_keys` 中移除：

In [9]:
param.parameter_keys()

dict_keys(['a', 'b'])

使用 `del_equal_constraints` 方法可删除已添加的等式约束，相关参数会重新出现在 `parameter_keys` 中，恢复原状态：

In [10]:
param.del_equal_constraints("c")

param.parameter_keys()
print(param['c'],param['c'].lb)

1.0 0.1


## 参数优先级

在 `Parameters` 类中，参数分为基本参数、约束参数、衍生参数和额外信息。  
它们的优先级从高到低依次为：约束参数 > 基本参数 > 衍生参数 > 额外信息。  
即在获取某个参数时，会依次在 `constraint_keys`、`parameter_keys`、`derived_keys`、`info_keys` 中查找。

示例如下：

建立参数集并添加额外信息 `eps_ab`：

In [11]:
params = Parameters(a=2, b=1)
params.add_info(eps_ab=5)
params['eps_ab']

5

添加衍生参数 `eps_ab`，此时获取到的是衍生参数的值：

In [12]:
params.add_derived("eps_ab", lambda p: 1 - p['b']/p['a'])
params['eps_ab']

0.5

直接添加基本参数 `eps_ab`，此时获取到的是基本参数的值：

In [13]:
params['eps_ab'] = 0.4
params['eps_ab']

0.4

添加约束参数 `eps_ab`，此时获取到的是约束参数的值：

In [14]:
params.add_equal_constraints(eps_ab=lambda k: 0.25)
params['eps_ab']

0.25

移除约束后，获取到的是基本参数的值：

In [15]:
params.del_equal_constraints("eps_ab")
params['eps_ab']

0.4